# 실시간 문장 모호성 특징 추출기 (Real-time Sentence Ambiguity Feature Extractor)

본 노트북은 VLM 다중 이미지 순서 정렬 파이프라인 아키텍처의 **Phase 1(통사 구조 분류)** 및 **Phase 2(경량 의미론적 특징 추출)** 설계에 맞추어, 임의의 입력 문장에 대한 문법적 특징을 실시간으로 분석하고 추출합니다.

### 추출되는 3대 핵심 특징:
1. **Partition (통사 구조 유형)**: `Type-1` (단일 절), `Type-2` (복합 종속), `Type-3` (대등 병렬)
2. **Pred_Count (서술어 개수)**: 문장 내 동사 및 조동사 (`VERB`, `AUX`) 개수
3. **Subj_Count (주어 개수)**: 문장 내 행위 주체 주어 (`nsubj`, `nsubjpass` 등) 개수

In [ ]:
# 1. 환경 설정 및 SpaCy 모델 로드
try:
    import spacy
except ImportError:
    import subprocess
    import sys
    print("SpaCy 패키지가 없어 설치를 진행합니다...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "spacy"])
    import spacy

# 영어 경량 의존성 구문 분석 모델 로드
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("en_core_web_sm 모델 다운로드 중...")
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

print("SpaCy 환경 로드 완료!")

## 2. 특징 추출 함수 정의 (일반화된 SpaCy 로직)

`train.csv`에 컬럼을 추가할 때와 동일하게 SpaCy 의존성 구문 트리(`dep_`)와 품사 태그(`pos_`)를 이용해 문법 요소를 계산합니다.

In [ ]:
def extract_ambiguity_features(sentence):
    """
    임의의 영문 문장을 분석하여 통사 유형(Partition), 서술어 개수, 주어 개수를 반환합니다.
    """
    if not isinstance(sentence, str) or not sentence.strip():
        return {
            "Partition": "Type-1",
            "Pred_Count": 0,
            "Subj_Count": 0
        }
    
    doc = nlp(sentence)
    
    # 1. 통사 구조 분류 (Type-1, 2, 3)
    has_subordinate_clause = False  # advcl (부사절 종속) 또는 ccomp (절 보충) 유무
    has_parallel_clause = False     # conj (등위절 결합) 동사/조동사 유무
    
    for token in doc:
        if token.dep_ in {"advcl", "ccomp"}:
            has_subordinate_clause = True
        if token.dep_ == "conj" and token.pos_ in {"VERB", "AUX"}:
            has_parallel_clause = True
            
    if not has_subordinate_clause and not has_parallel_clause:
        partition = "Type-1"
    elif has_subordinate_clause:
        partition = "Type-2"
    else:
        partition = "Type-3"
        
    # 2. 주어(Subject) 및 서술어(Predicate) 카운트
    subj_deps = {"nsubj", "nsubjpass", "csubj", "csubjpass"}
    subjs = [t for t in doc if t.dep_ in subj_deps]
    preds = [t for t in doc if t.pos_ in {"VERB", "AUX"}]
    
    return {
        "Partition": partition,
        "Pred_Count": len(preds),
        "Subj_Count": len(subjs)
    }

## 3. 실시간 단일 문장 테스트

원하는 문장을 아래 `test_sentence`에 입력하여 실시간으로 도출되는 3대 특징 결과를 확인해 볼 수 있습니다.

In [ ]:
# 다양한 통사 구조 예시 테스트
test_sentences = [
    "A dog jumps over the fence.",  # Type-1 (단일 절)
    "While the man is cutting a paper, the woman writes on a whiteboard.",  # Type-2 (복합 종속)
    "The boy kicks the ball, runs to the goalpost, and celebrates.",  # Type-3 (대등 병렬)
    "First a camera zooms out to reveal a living room, then a cat runs under the table." # Type-2
]

print("=== 테스트 문장 분석 결과 ===")
for s in test_sentences:
    res = extract_ambiguity_features(s)
    print(f"\nSentence: \"{s}\"")
    print(f" -> Partition  : {res['Partition']}")
    print(f" -> Pred_Count : {res['Pred_Count']}")
    print(f" -> Subj_Count : {res['Subj_Count']}")

### 사용자 맞춤 입력 테스트
원하는 문장을 적고 실행해보세요!

In [ ]:
my_sentence = "First the camera zooms in on the player's face, then the scene shifts to a wide shot of the stadium."
features = extract_ambiguity_features(my_sentence)

print(f"입력 문장: \"{my_sentence}\"")
print("추출된 특징:", features)

## 4. 실시간 대용량 배치 처리 예제 (`nlp.pipe` 사용)

많은 문장 데이터를 한 번에 빠르게 실시간 처리할 때 속도를 보장하기 위해 `nlp.pipe`를 활용해 멀티 배치 형태로 연산하는 코드입니다.

In [ ]:
def extract_features_batch(sentences, batch_size=256):
    """
    배치 연산을 통해 대량의 문장 데이터를 빠르게 가공합니다.
    """
    results = []
    subj_deps = {"nsubj", "nsubjpass", "csubj", "csubjpass"}
    
    # nlp.pipe를 이용한 일괄 처리
    for doc in nlp.pipe(sentences, batch_size=batch_size):
        has_subordinate_clause = False
        has_parallel_clause = False
        
        for token in doc:
            if token.dep_ in {"advcl", "ccomp"}:
                has_subordinate_clause = True
            if token.dep_ == "conj" and token.pos_ in {"VERB", "AUX"}:
                has_parallel_clause = True
                
        if not has_subordinate_clause and not has_parallel_clause:
            partition = "Type-1"
        elif has_subordinate_clause:
            partition = "Type-2"
        else:
            partition = "Type-3"
            
        subjs = [t for t in doc if t.dep_ in subj_deps]
        preds = [t for t in doc if t.pos_ in {"VERB", "AUX"}]
        
        results.append({
            "Partition": partition,
            "Pred_Count": len(preds),
            "Subj_Count": len(subjs)
        })
    return results

# 배치 테스트 실행
many_sentences = [
    "A skater slips on the ice.",
    "The chef chops onions and mixes them with garlic.",
    "While the engine starts, the pilot checks the instruments.",
    "The singer holds the microphone, walks to the stage, and sings."
] * 10  # 40개 문장 데이터 생성

batch_results = extract_features_batch(many_sentences)
print(f"배치 처리 완료! 총 {len(batch_results)}개 중 최초 3개 샘플:")
for i in range(3):
    print(f"문장 {i+1}: \"{many_sentences[i]}\" -> {batch_results[i]}")